# 🚗 Philippine LPRNet Character Recognition Training & Evaluation
> **Thesis Project:** License Plate Recognition Based Vehicle Logging System for Lessandra Subdivision, Phase 2, Salinas 1, Bacoor, Cavite  
> **Camera:** TP-Link Tapo C260  
> **Target:** Eliminate Z vs 2, B vs 8, D vs 0, S vs 5 character confusions using LPRNet CTC loss.


### Step 1: Verify NVIDIA T4 GPU Allocation


In [ ]:
!nvidia-smi


### Step 2: Clone or Prepare Project Files & Install Dependencies


In [ ]:
!pip install -q easyocr paddlepaddle paddleocr opencv-python pyyaml tqdm

import os
%cd /content
if not os.path.exists('/content/NewCapstone'):
    !git clone https://github.com/THU-MIG/yolov10.git /content/NewCapstone 2>/dev/null || true


### Step 3: Upload Labeled Philippine Plate Dataset
Upload your ph_plates.zip containing nnotations.csv and images/ folder.


In [ ]:
import os
import shutil
from google.colab import files

if not os.path.exists('/content/ph_plates'):
    print('Please upload your ph_plates.zip dataset:')
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            !unzip -q {fname} -d /content/ph_plates
            print('Extracted dataset to /content/ph_plates')


### Step 4: Train LPRNet with CTC Loss & Confusable Tracking


In [ ]:
# Train LPRNet on Philippine plates with Cosine Annealing and Z/2 monitoring
!python /content/NewCapstone/train_lprnet_ph.py \
    --data-dir /content/ph_plates \
    --epochs 80 \
    --batch-size 32 \
    --lr 0.001 \
    --device cuda


### Step 5: Run 3-Way Comparative Benchmark (Section 6 Thesis Evaluation)
Evaluates EasyOCR vs PaddleOCR vs LPRNet on the exact same held-out validation set.


In [ ]:
!python /content/NewCapstone/evaluate_ocr_models.py \
    --val-dir /content/ph_plates \
    --weights /content/NewCapstone/weights/lprnet_ph_best.pt \
    --output-report /content/thesis_evaluation.json


### Step 6: Download Trained LPRNet Weights (lprnet_ph_best.pt)


In [ ]:
from google.colab import files
best_model_path = '/content/NewCapstone/weights/lprnet_ph_best.pt'
if os.path.exists(best_model_path):
    print('Downloading trained LPRNet weights to your PC...')
    files.download(best_model_path)
    if os.path.exists('/content/thesis_evaluation.json'):
        files.download('/content/thesis_evaluation.json')
else:
    print('Trained weights not found.')
